# NutriSnap — Plan A Integration Spike

**Goal:** Determine whether `hsilvosa/nova-group-classifier` (pinned revision) can be
used as-is, and which pooling strategy the `nova_gb_model.joblib` head expects.

**Spec:** ARCHITECTURE.md §7.2
**Time-box:** 2 hours. If any step fails, document it and switch to Plan B.

**Pinned revision:** `f252e3546650954596ff7e5926aab4559c433909`

**Record your results in `docs/decisions.md` before committing.**

---
Accept Plan A only if:
- Chosen pooling gets ≥ 9 / 12 sanity strings correct, AND
- Median top-1 probability on the sanity set ≥ 0.70

Otherwise switch to Plan B (MiniLM + own head — see `train_head_colab.ipynb`).

## Step 0 — Install dependencies

In [ ]:
# Run once per Colab session.
!pip install -q huggingface_hub transformers torch scikit-learn joblib numpy

## Step 1 — Download pinned revision

In [ ]:
from huggingface_hub import snapshot_download

REPO_ID   = "hsilvosa/nova-group-classifier"
REVISION  = "f252e3546650954596ff7e5926aab4559c433909"  # pinned — never change
LOCAL_DIR = "/content/hsilvosa-nova"

print(f"Downloading {REPO_ID} @ {REVISION[:8]} …")
model_path = snapshot_download(
    repo_id=REPO_ID,
    revision=REVISION,
    local_dir=LOCAL_DIR,
)
print(f"Downloaded to: {model_path}")

## Step 2 — Inspect config.json and repo README

In [ ]:
import json, pathlib

config_path = pathlib.Path(LOCAL_DIR) / "config.json"
with open(config_path) as f:
    config = json.load(f)

model_type  = config.get("model_type", "UNKNOWN")
hidden_size = config.get("hidden_size", "UNKNOWN")

print(f"model_type  : {model_type}")
print(f"hidden_size : {hidden_size}")
print("\nFull config:")
print(json.dumps(config, indent=2))

In [ ]:
# Print the model card README so we can inspect the example code.
readme_path = pathlib.Path(LOCAL_DIR) / "README.md"
if readme_path.exists():
    print(readme_path.read_text())
else:
    print("No README.md found in the repo.")

## Step 3 — Load the joblib head and inspect it

In [ ]:
# SECURITY NOTE: Only load pickles from the pinned revision in a project-controlled
# folder. Never load user-supplied joblib files.
import joblib

joblib_path = pathlib.Path(LOCAL_DIR) / "nova_gb_model.joblib"
head = joblib.load(joblib_path)

print(f"Type            : {type(head)}")
print(f"n_features_in_  : {head.n_features_in_}")

if hasattr(head, 'classes_'):
    print(f"classes_        : {head.classes_}")
if hasattr(head, 'n_estimators'):
    print(f"n_estimators    : {head.n_estimators}")

print(f"\nhidden_size     : {hidden_size}")
print(f"2 * hidden_size : {2 * hidden_size if isinstance(hidden_size, int) else 'N/A'}")

# Decision rule per ARCHITECTURE §7.2 step 5:
if isinstance(hidden_size, int):
    if head.n_features_in_ == hidden_size:
        print("\n>>> DECISION: CLS or mean pooling alone (single vector). Try both.")
    elif head.n_features_in_ == 2 * hidden_size:
        print("\n>>> DECISION: CLS + mean concatenation (two vectors joined). Try that.")
    else:
        print(f"\n>>> DECISION: n_features_in_ ({head.n_features_in_}) matches neither hidden_size nor 2*hidden_size.")
        print("    ABANDON Plan A → start Plan B immediately.")

## Step 4 — Load tokenizer and encoder; compute CLS and mean poolings

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

print(f"torch version      : {torch.__version__}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device             : {device}")

tokenizer = AutoTokenizer.from_pretrained(LOCAL_DIR)
encoder   = AutoModel.from_pretrained(LOCAL_DIR)
encoder   = encoder.to(device)
encoder.eval()
torch.set_num_threads(2)

print("Tokenizer and encoder loaded.")

In [ ]:
import numpy as np

def embed(text: str) -> dict:
    """
    Tokenize text and return CLS embedding, masked-mean embedding, and their
    concatenation. All returned as numpy float32 arrays.
    Spec: ARCHITECTURE §7.2 step 4.
    """
    inputs = tokenizer(
        text,
        return_tensors="pt",
        max_length=256,
        truncation=True,
        padding=True,
    ).to(device)

    with torch.inference_mode():
        outputs = encoder(**inputs)

    last_hidden = outputs.last_hidden_state  # (1, seq_len, hidden)

    # CLS token (position 0).
    cls_vec = last_hidden[0, 0, :].cpu().numpy().astype(np.float32)  # (hidden,)

    # Masked mean pooling (exclude padding tokens).
    attention_mask = inputs["attention_mask"]  # (1, seq_len)
    mask_expanded  = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()
    sum_hidden     = (last_hidden * mask_expanded).sum(dim=1)
    count          = mask_expanded.sum(dim=1).clamp(min=1e-9)
    mean_vec       = (sum_hidden / count)[0].cpu().numpy().astype(np.float32)  # (hidden,)

    # Concatenation.
    concat_vec = np.concatenate([cls_vec, mean_vec], axis=0)  # (2*hidden,)

    return {"cls": cls_vec, "mean": mean_vec, "concat": concat_vec}

# Quick smoke-test.
test_emb = embed("wheat, sugar, salt")
print(f"CLS shape    : {test_emb['cls'].shape}")
print(f"Mean shape   : {test_emb['mean'].shape}")
print(f"Concat shape : {test_emb['concat'].shape}")

## Step 5 — Decision: which pooling does the head expect?

In [ ]:
# Choose the feature vector that matches n_features_in_.
n_feat = head.n_features_in_

if n_feat == test_emb["cls"].shape[0]:
    pooling = "cls"
elif n_feat == test_emb["mean"].shape[0]:
    pooling = "mean"
elif n_feat == test_emb["concat"].shape[0]:
    pooling = "concat"
else:
    pooling = None
    print(f"ERROR: n_features_in_={n_feat} does not match any pooling strategy.")
    print("→ ABANDON Plan A. Start Plan B.")

if pooling:
    print(f"Selected pooling: {pooling}  (n_features_in_={n_feat})")

## Step 6 — Sanity set (12 strings, expected NOVA in brackets)

In [ ]:
# Sanity set from ARCHITECTURE §7.2 step 6.
# Expected NOVA groups are from the NOVA definitions on the model card.
SANITY_SET = [
    # NOVA 1 — unprocessed / minimally processed
    ("wheat", 1),
    ("whole milk", 1),
    # NOVA 2 — processed culinary ingredients
    ("sugar", 2),
    ("refined palm oil", 2),
    ("salt", 2),
    # NOVA 3 — processed foods
    ("wheat flour, water, yeast, salt, sugar, vegetable oil", 3),
    ("chickpeas, water, salt", 3),
    # NOVA 4 — ultra-processed
    ("wheat flour, palm oil, sugar, salt, maltodextrin, artificial flavour, monosodium glutamate, tbhq", 4),
    ("refined wheat flour, sugar, hydrogenated vegetable fat, glucose syrup, artificial flavour, tartrazine, sodium benzoate", 4),
    ("carbonated water, sugar, caramel colour, phosphoric acid, natural flavours, caffeine", 4),
    ("potato, palm oil, sugar, salt, artificial flavour, monosodium glutamate, citric acid, tbhq", 4),
    ("skimmed milk, sugar, water, modified starch, artificial flavour, carrageenan, sodium benzoate", 4),
]

print(f"{'#':>2}  {'Expected':>8}  {'Predicted':>9}  {'Confidence':>10}  {'Correct':>7}  Text")
print("-" * 100)

results = []
for i, (text, expected_nova) in enumerate(SANITY_SET, 1):
    if pooling is None:
        break

    emb  = embed(text)[pooling]
    proba = head.predict_proba([emb])[0]        # (n_classes,)
    classes = list(head.classes_)               # e.g. [1, 2, 3, 4] or ['1','2','3','4']
    pred_idx = int(np.argmax(proba))
    pred_nova = int(classes[pred_idx])
    confidence = float(proba[pred_idx])

    correct = (pred_nova == expected_nova)
    results.append({"expected": expected_nova, "predicted": pred_nova, "confidence": confidence, "correct": correct})

    print(f"{i:>2}  {expected_nova:>8}  {pred_nova:>9}  {confidence:>10.3f}  {'✓' if correct else '✗':>7}  {text[:60]}")

if results:
    n_correct = sum(r["correct"] for r in results)
    median_conf = float(np.median([r["confidence"] for r in results]))
    print(f"\nCorrect: {n_correct} / {len(results)}")
    print(f"Median top-1 confidence: {median_conf:.3f}")

## Step 7 — Accept / Reject Plan A

In [ ]:
# Acceptance criteria (ARCHITECTURE §7.2 step 7):
#   - At least 9 / 12 correct, AND
#   - Median top-1 probability >= 0.70

if not results or pooling is None:
    print("PLAN A REJECTED — pooling could not be determined.")
    print("→ Switch to Plan B immediately (train_head_colab.ipynb).")
else:
    n_correct  = sum(r["correct"] for r in results)
    median_conf = float(np.median([r["confidence"] for r in results]))
    threshold_correct = 9
    threshold_conf    = 0.70

    passed = (n_correct >= threshold_correct) and (median_conf >= threshold_conf)

    print("=" * 60)
    if passed:
        print("PLAN A ACCEPTED ✓")
        print(f"  Pooling strategy : {pooling}")
        print(f"  Correct          : {n_correct} / {len(results)} (need >= {threshold_correct})")
        print(f"  Median confidence: {median_conf:.3f} (need >= {threshold_conf})")
        print()
        print("Next steps:")
        print("  1. Run the Indian-subset evaluation (Step 8).")
        print("  2. Record pooling strategy and results in docs/decisions.md.")
        print("  3. Implement app/core/ml/hf_nova.py in M3.")
    else:
        print("PLAN A REJECTED ✗")
        print(f"  Correct          : {n_correct} / {len(results)} (need >= {threshold_correct})")
        print(f"  Median confidence: {median_conf:.3f} (need >= {threshold_conf})")
        print()
        print("→ Switch to Plan B: train_head_colab.ipynb")
    print("=" * 60)

## Step 8 — Indian-subset evaluation (run only if Plan A accepted)

Spec: ARCHITECTURE §7.5.  
Uses DuckDB to stream from the Open Food Facts HF dataset — **do not download the full file**.  
Filter: `countries_tags` contains `en:india`, `nova_group IS NOT NULL`, non-empty `ingredients_text`.

In [ ]:
# Install DuckDB for streaming parquet queries.
!pip install -q duckdb

In [ ]:
import duckdb
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# ── Query Indian rows from Open Food Facts (ODbL; attribution required) ──────
# IMPORTANT: Verify the schema first — ingredients_text may be a list of structs.
# The query below uses a simple string column; adapt if needed.

HF_PARQUET = "hf://datasets/openfoodfacts/product-database/food.parquet"

print("Inspecting dataset schema …")
schema_q = duckdb.sql(f"DESCRIBE SELECT * FROM read_parquet('{HF_PARQUET}') LIMIT 1")
schema_q.show()

# ── Adapt the column name for ingredients_text if needed ─────────────────────
# Common names: 'ingredients_text', 'ingredients_text_en', or a list column.
# Inspect and adjust the query below.

INGREDIENTS_COL = "ingredients_text"   # ← adjust if schema differs
NOVA_COL        = "nova_group"         # ← adjust if schema differs
COUNTRIES_COL   = "countries_tags"     # ← adjust if schema differs

query = f"""
SELECT
    {INGREDIENTS_COL} AS ingredients_text,
    CAST({NOVA_COL} AS INTEGER) AS nova_group
FROM read_parquet('{HF_PARQUET}')
WHERE
    list_contains({COUNTRIES_COL}, 'en:india')
    AND {NOVA_COL} IS NOT NULL
    AND {NOVA_COL} IN (1, 2, 3, 4)
    AND {INGREDIENTS_COL} IS NOT NULL
    AND LENGTH(TRIM({INGREDIENTS_COL})) > 10
LIMIT 5000
"""

print("Querying Indian products …")
df = duckdb.sql(query).df()
print(f"Rows fetched: {len(df)}")
print(df["nova_group"].value_counts().sort_index())

In [ ]:
# Skip if Plan A was rejected or too few rows.
if pooling is None or len(df) < 20:
    print("Skipping evaluation (Plan A rejected or insufficient data).")
else:
    # Gate thresholds (from ARCHITECTURE §7.3 / Appendix C defaults).
    ML_MIN_CONFIDENCE = 0.60
    ML_NOVA2_MIN_CONFIDENCE = 0.80
    ML_MIN_MARGIN = 0.15

    y_true, y_pred, confidences, gate_rejected = [], [], [], []

    classes = [int(c) for c in head.classes_]

    for _, row in df.iterrows():
        text = str(row["ingredients_text"]).strip()
        if not text:
            continue

        emb   = embed(text)[pooling]
        proba = head.predict_proba([emb])[0]

        sorted_idx  = np.argsort(proba)[::-1]
        top1_idx    = sorted_idx[0]
        top2_idx    = sorted_idx[1]
        pred_nova   = classes[top1_idx]
        confidence  = float(proba[top1_idx])
        margin      = float(proba[top1_idx] - proba[top2_idx])

        # Gate checks (confidence only — no rule score here).
        rejected = False
        if confidence < ML_MIN_CONFIDENCE:
            rejected = True
        elif pred_nova == 2 and confidence < ML_NOVA2_MIN_CONFIDENCE:
            rejected = True
        elif margin < ML_MIN_MARGIN:
            rejected = True

        gate_rejected.append(rejected)
        y_true.append(int(row["nova_group"]))
        y_pred.append(pred_nova)
        confidences.append(confidence)

    # ── Metrics ──────────────────────────────────────────────────────────────
    fallback_rate = sum(gate_rejected) / len(gate_rejected)

    print("\n" + "=" * 60)
    print(f"Indian-subset evaluation  (n={len(y_true)})")
    print("=" * 60)
    print(f"Accuracy      : {accuracy_score(y_true, y_pred):.4f}")
    print(f"Macro-F1      : {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"Gate fallback : {fallback_rate:.2%} of rows rejected by confidence gate")

    # Majority baseline.
    majority_class = max(set(y_true), key=y_true.count)
    majority_acc   = y_true.count(majority_class) / len(y_true)
    print(f"Majority class: NOVA {majority_class} ({majority_acc:.2%} accuracy — baseline)")

    print("\nClassification report:")
    print(classification_report(y_true, y_pred, zero_division=0))

    print("\nConfusion matrix (rows=true, cols=pred, groups 1-4):")
    print(confusion_matrix(y_true, y_pred, labels=[1, 2, 3, 4]))

In [ ]:
# ── Reliability table (10-bin calibration) ───────────────────────────────────
if 'y_true' in dir() and len(y_true) >= 20:
    n_bins = 10
    confs  = np.array(confidences)
    correct_arr = np.array(y_true) == np.array(y_pred)

    print("\nReliability table (confidence bins vs actual accuracy):")
    print(f"{'Bin':>12}  {'N':>6}  {'Avg conf':>10}  {'Actual acc':>12}")
    print("-" * 50)
    bins = np.linspace(0, 1, n_bins + 1)
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (confs >= lo) & (confs < hi)
        n    = mask.sum()
        if n == 0:
            continue
        avg_conf   = confs[mask].mean()
        avg_acc    = correct_arr[mask].mean()
        print(f"[{lo:.1f}, {hi:.1f})  {n:>6}  {avg_conf:>10.3f}  {avg_acc:>12.3f}")

    print()
    print("If the model is badly over-confident, raise ML_MIN_CONFIDENCE in .env.")

## Step 9 — Record results in docs/decisions.md

In [ ]:
print("""
Fill in docs/decisions.md with:

## Spike result (Plan A — hsilvosa/nova-group-classifier)

- Revision : f252e3546650954596ff7e5926aab4559c433909
- hidden_size  : <value from Step 2>
- n_features_in_ : <value from Step 3>
- Pooling chosen : <cls | mean | concat | N/A — Plan B>
- Sanity set     : <n_correct> / 12 correct, median confidence <median_conf>
- Plan A accepted: <yes | no>
- Indian subset  : n=<n>, accuracy=<acc>, macro-F1=<f1>, gate fallback=<rate>
- Decision       : Use <Plan A with <pooling> pooling | Plan B MiniLM head>
- Calibration note: <over-confident? raise ML_MIN_CONFIDENCE?>
""")